In [1]:
# === KAGGLE NOTEBOOK: Two-Stage Training (Corrected) ===
# Upload: train_df.parquet, val_df.parquet, test_df.parquet, dataset_metadata.pkl

!pip install transformers -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, label_ranking_average_precision_score
from tqdm import tqdm

# --- Device setup ---
MULTI_GPU = torch.cuda.device_count() > 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | GPUs: {torch.cuda.device_count()}")

# --- Load data ---
DATA_DIR = "/kaggle/input/datasets/ogoud073/new-cve2attack"
train_df = pd.read_parquet(f"{DATA_DIR}/train_df.parquet")
val_df = pd.read_parquet(f"{DATA_DIR}/val_df.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test_df.parquet")

with open(f"{DATA_DIR}/dataset_metadata.pkl", "rb") as f:
    meta = pickle.load(f)

tech2idx = meta["tech2idx"]
idx2tech = meta["idx2tech"]
NUM_CLASSES = meta["NUM_CLASSES"]
MODEL_NAME = meta.get("MODEL_NAME", "ehsanaghaei/SecureBERTPlus")
MAX_LEN = meta.get("MAX_LEN", 512)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Classes: {NUM_CLASSES} | Model: {MODEL_NAME} | MaxLen: {MAX_LEN}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Dataset (uses 'description' from parquet, no CVSS fields) ---
class CVEDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, tech2idx, num_classes):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.tech2idx = tech2idx
        self.num_classes = num_classes

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["description"]
        encoding = self.tokenizer(text, max_length=self.max_len, padding="max_length",
                                  truncation=True, return_tensors="pt")
        label = torch.zeros(self.num_classes)
        for t in row["techniques"]:
            if t in self.tech2idx:
                label[self.tech2idx[t]] = 1.0
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": label,
        }

# --- Model ---
class SecureBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(outputs.last_hidden_state[:, 0, :]))

# --- ASL Loss ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-6):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, labels):
        probs = torch.sigmoid(logits).clamp(self.eps, 1.0 - self.eps)
        probs_neg = (probs + self.clip).clamp(max=1.0 - self.eps)
        pos_w = (1.0 - probs).detach() ** self.gamma_pos
        neg_w = probs_neg.detach() ** self.gamma_neg
        loss = (-labels * pos_w * torch.log(probs)
                -(1 - labels) * neg_w * torch.log(1 - probs_neg)).clamp(max=10.0)
        return loss.mean()

# --- Evaluate ---
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labs = batch["labels"].to(device)
            logits = model(ids, mask) if not MULTI_GPU else model(ids, mask)
            total_loss += criterion(logits, labs).item() * len(labs)
            p = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(p)
            all_preds.append((p > threshold).astype(int))
            all_labels.append(labs.cpu().numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    all_probs = np.vstack(all_probs)
    micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    has_label = all_labels.sum(axis=1) > 0
    lrap = label_ranking_average_precision_score(all_labels[has_label], all_probs[has_label])
    return total_loss / len(loader.dataset), micro, macro, lrap

# ============================================
# STAGE 1: Pretrain on ALL data
# ============================================
print("=" * 60)
print("STAGE 1: Pretraining on all data")
print("=" * 60)

train_dataset = CVEDataset(train_df, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)
val_dataset = CVEDataset(val_df, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                        num_workers=2, pin_memory=True)

model = SecureBERTClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
if MULTI_GPU:
    model = nn.DataParallel(model)

criterion = AsymmetricLoss()
EPOCHS_S1 = 5
GRAD_ACCUM = 2
LR = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS_S1 // GRAD_ACCUM
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)

best_micro = 0
print(f"\n{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'µF1':>6} | {'MF1':>6} | {'LRAP':>6}")
print("-" * 55)

for epoch in range(EPOCHS_S1):
    model.train()
    ep_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(train_loader, leave=False, desc=f"S1 Ep{epoch+1}")):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        loss = criterion(model(ids, mask), labs) / GRAD_ACCUM
        loss.backward()
        ep_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    vl, vm, vmac, vlrap = evaluate(model, val_loader, criterion, DEVICE)
    is_best = vm > best_micro
    if is_best:
        best_micro = vm
        state_dict = model.module.state_dict() if MULTI_GPU else model.state_dict()
        torch.save(state_dict, "stage1_best.pt")
    print(f"{epoch+1:>3} | {ep_loss/len(train_loader):>8.4f} | {vl:>8.4f} | {vm:>6.4f} | {vmac:>6.4f} | {vlrap:>6.4f} {'✓' if is_best else ''}")

print(f"\nStage 1 best µF1: {best_micro:.4f}")

# ============================================
# STAGE 2: Fine-tune on gold data
# ============================================
print("\n" + "=" * 60)
print("STAGE 2: Fine-tuning on gold-labeled data")
print("=" * 60)

# Fresh model, load stage 1 weights (before DataParallel)
model = SecureBERTClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load("stage1_best.pt",weights_only=False))

# Freeze embeddings + first 6 encoder layers (before DataParallel)
for param in model.bert.embeddings.parameters():
    param.requires_grad = False
for i in range(6):
    for param in model.bert.encoder.layer[i].parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# Wrap in DataParallel after freezing
if MULTI_GPU:
    model = nn.DataParallel(model)

# Filter gold tiers
gold_train = train_df[train_df["tier"].isin(["gold", "gold+transitive"])].copy()
gold_val = val_df[val_df["tier"].isin(["gold", "gold+transitive"])].copy()
print(f"Gold train: {len(gold_train):,} | Gold val: {len(gold_val):,}")

# If gold too small, add sampled transitive data
if len(gold_train) < 500:
    print("Gold set too small, adding sampled transitive data...")
    n_extra = min(2000, len(train_df[train_df["tier"] == "transitive"]))
    trans_sample = train_df[train_df["tier"] == "transitive"].sample(n=n_extra, random_state=42)
    gold_train = pd.concat([gold_train, trans_sample]).reset_index(drop=True)
    n_val_extra = min(500, len(val_df[val_df["tier"] == "transitive"]))
    trans_val = val_df[val_df["tier"] == "transitive"].sample(n=n_val_extra, random_state=42)
    gold_val = pd.concat([gold_val, trans_val]).reset_index(drop=True)
    print(f"Expanded: train={len(gold_train):,} | val={len(gold_val):,}")

gold_train_ds = CVEDataset(gold_train, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)
gold_val_ds = CVEDataset(gold_val, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)

BS2 = min(16, len(gold_train_ds))
gold_train_loader = DataLoader(gold_train_ds, batch_size=BS2, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True)
gold_val_loader = DataLoader(gold_val_ds, batch_size=BS2 * 2, shuffle=False,
                             num_workers=2, pin_memory=True)

EPOCHS_S2 = 20
LR2 = 5e-6
optimizer2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR2, weight_decay=0.01)
total_steps2 = len(gold_train_loader) * EPOCHS_S2
scheduler2 = get_linear_schedule_with_warmup(optimizer2, int(total_steps2 * 0.1), total_steps2)

best_gold_lrap = 0
best_gold_epoch = 0
print(f"\n{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'µF1':>6} | {'MF1':>6} | {'LRAP':>6} | {'Full µF1':>8}")
print("-" * 70)

for epoch in range(EPOCHS_S2):
    model.train()
    ep_loss = 0
    for batch in tqdm(gold_train_loader, leave=False, desc=f"S2 Ep{epoch+1}"):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        optimizer2.zero_grad()
        loss = criterion(model(ids, mask), labs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer2.step()
        scheduler2.step()
        ep_loss += loss.item()

    vl, vm, vmac, vlrap = evaluate(model, gold_val_loader, criterion, DEVICE)
    _, fvm, fvmac, fvlrap = evaluate(model, val_loader, criterion, DEVICE)

    is_best = vlrap > best_gold_lrap
    if is_best:
        best_gold_lrap = vlrap
        best_gold_epoch = epoch + 1
        state_dict = model.module.state_dict() if MULTI_GPU else model.state_dict()
        torch.save({
            "model_state_dict": state_dict,
            "epoch": epoch + 1,
            "gold_lrap": vlrap,
            "full_micro_f1": fvm,
            "tech2idx": tech2idx,
            "idx2tech": idx2tech,
            "NUM_CLASSES": NUM_CLASSES,
            "MODEL_NAME": MODEL_NAME,
            "MAX_LEN": MAX_LEN,
            "approach": "two_stage",
        }, "best_two_stage.pt")

    if (epoch + 1) % 5 == 0 or is_best:
        print(f"{epoch+1:>3} | {ep_loss/len(gold_train_loader):>8.4f} | {vl:>8.4f} | {vm:>6.4f} | {vmac:>6.4f} | {vlrap:>6.4f} | {fvm:>8.4f} {'✓' if is_best else ''}")

print(f"\nBest Stage 2: epoch {best_gold_epoch}, gold LRAP={best_gold_lrap:.4f}")

# ============================================
# FINAL EVALUATION ON TEST SET
# ============================================
print("\n" + "=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

# Load best two-stage model
model = SecureBERTClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
checkpoint = torch.load("best_two_stage.pt", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
if MULTI_GPU:
    model = nn.DataParallel(model)

test_dataset = CVEDataset(test_df, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                         num_workers=2, pin_memory=True)

tl, tm, tmac, tlrap = evaluate(model, test_loader, criterion, DEVICE)
print(f"Test Loss: {tl:.4f}")
print(f"Test µF1:  {tm:.4f}")
print(f"Test MF1:  {tmac:.4f}")
print(f"Test LRAP: {tlrap:.4f}")

# Per-tier evaluation
print("\n--- Per-Tier Test Results ---")
for tier in test_df["tier"].unique():
    tier_df = test_df[test_df["tier"] == tier]
    if len(tier_df) < 5:
        continue
    tier_ds = CVEDataset(tier_df, tokenizer, MAX_LEN, tech2idx, NUM_CLASSES)
    tier_loader = DataLoader(tier_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             num_workers=2, pin_memory=True)
    _, t_micro, t_macro, t_lrap = evaluate(model, tier_loader, criterion, DEVICE)
    print(f"  {tier:20s}: n={len(tier_df):>5,} | µF1={t_micro:.4f} | MF1={t_macro:.4f} | LRAP={t_lrap:.4f}")

# Save final model
torch.save(checkpoint, "final_two_stage_model.pt")
print("\n✅ Saved: final_two_stage_model.pt")

Device: cuda | GPUs: 2
Train: 62,087 | Val: 7,761 | Test: 7,761
Classes: 137 | Model: ehsanaghaei/SecureBERT | MaxLen: 512


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

 15 |   0.0114 |   0.0144 | 0.5889 | 0.2051 | 0.7498 |   0.6318 


 20 |   0.0110 |   0.0143 | 0.5944 | 0.2165 | 0.7499 |   0.6343 

Best Stage 2: epoch 13, gold LRAP=0.7538

FINAL EVALUATION ON TEST SET


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ehsanaghaei/SecureBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
    response.raise_for_st

Test Loss: 0.0123
Test µF1:  0.6351
Test MF1:  0.2286
Test LRAP: 0.7879

--- Per-Tier Test Results ---
  transitive          : n=7,729 | µF1=0.6367 | MF1=0.2262 | LRAP=0.7894
  gold+transitive     : n=   10 | µF1=0.5806 | MF1=0.0970 | LRAP=0.6764
  gold                : n=   22 | µF1=0.1706 | MF1=0.0285 | LRAP=0.2822

✅ Saved: final_two_stage_model.pt
